# Cumulative Spend Distribution Analysis

This notebook characterizes the underlying cumulative spend-position distribution that feeds the later Beta CDF modeling work. It should be read before `clustered_curve_beta_cdf_model.ipynb`: this page explains the empirical shape of `CUMULATIVEBURNPCT` as a function of `ELAPSEDPCT`, while the later notebook evaluates predictive reference curves against held-out items.

## Dataset and Overall Character

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_adams.csv |
| Cluster curve rows | 606 |
| Items | 45 |
| Train rows used for distribution fitting | 414 |
| Train items | 35 |
| Mean cumulative spend pct | 55.17% |
| Median cumulative spend pct | 56.23% |
| Std dev cumulative spend pct | 31.96% |
| Share at completed edge near 100% | 9.74% |
| Share at zero edge near 0% | 0.00% |
| Pearson corr elapsed vs cumulative spend | 0.8583 |
| Spearman corr elapsed vs cumulative spend | 0.8603 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Marginal Distribution of Cumulative Spend

The marginal distribution is bounded between 0 and 1 and is strongly affected by repeated observations from the same item curve. It is not a standalone time-free distribution: a point at 90% elapsed and a point at 20% elapsed should not be expected to have the same cumulative spend behavior. The visible pile-up near 100% is expected because every completed item contributes a final cluster at full cumulative spend.

| Quantile | Cumulative spend pct |
| --- | --- |
| p01 | 1.16% |
| p05 | 4.44% |
| p10 | 9.21% |
| p25 | 27.21% |
| p50 | 56.23% |
| p75 | 84.72% |
| p90 | 99.42% |
| p95 | 100.00% |
| p99 | 100.00% |



## Joint Distribution With Elapsed Percent

The joint view is the core characterization. The distribution is bounded, monotonic in expectation, heteroscedastic, and edge-inflated near completion. The diagonal line is the pure linear spend reference; density above the line is front-loaded spend, while density below it is back-loaded or delayed spend.



## Conditional Distribution by Elapsed Bucket

The table and band plot summarize `CUMULATIVEBURNPCT | ELAPSEDPCT bucket`. The widening and narrowing of the percentile bands matters more than the overall histogram because the production question is where an item sits relative to expected cumulative spend at its current elapsed position.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR | Skew |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0-0.1 | 30 | 8.95% | 5.20% | 1.06% | 2.35% | 8.17% | 22.82% | 5.82% | 2.364 |
| 0.1-0.2 | 41 | 17.89% | 12.24% | 4.01% | 6.18% | 21.40% | 46.31% | 15.21% | 1.764 |
| 0.2-0.3 | 41 | 27.93% | 25.28% | 8.85% | 12.00% | 34.96% | 53.05% | 22.95% | 1.014 |
| 0.3-0.4 | 41 | 40.02% | 34.41% | 13.03% | 21.97% | 56.10% | 78.22% | 34.13% | 0.764 |
| 0.4-0.5 | 44 | 48.30% | 45.54% | 20.59% | 30.97% | 60.92% | 87.37% | 29.94% | 0.653 |
| 0.5-0.6 | 48 | 59.12% | 58.86% | 37.98% | 46.56% | 69.00% | 79.00% | 22.44% | 0.370 |
| 0.6-0.7 | 54 | 72.44% | 73.91% | 50.74% | 61.62% | 84.14% | 97.97% | 22.51% | -0.572 |
| 0.7-0.8 | 31 | 77.09% | 76.94% | 57.36% | 70.25% | 85.11% | 91.40% | 14.86% | -0.484 |
| 0.8-0.9 | 29 | 88.23% | 91.00% | 75.51% | 87.00% | 93.31% | 94.92% | 6.31% | -1.171 |
| 0.9-1.0 | 55 | 97.72% | 100.00% | 93.44% | 97.18% | 100.00% | 100.00% | 2.82% | -3.406 |

## Percentile Bands and Reference Curves

The empirical median curve is generally above the linear reference for much of the item life, which means these clustered payment curves are often front-loaded relative to time. The Beta CDF and anchored polynomial are compact parameterizations of that empirical shape; the later model notebook evaluates the Beta CDF approach more formally.



## Bucketed Density View

This view shows the full shape within each elapsed bucket rather than only percentiles. It makes the conditional spread visible: some elapsed regions are broad and multi-modal because items can receive lumpy clustered payments at different points in their lifecycle.



## Curve Parameterizations

| Model | Alpha / coefficients | Beta | MAE | RMSE | Bias | Clip share | Monotonic violations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Beta CDF | 0.979780 | 1.124220 | 0.1191 | 0.1691 | 0.0004 |  |  |
| Anchored polynomial degree 3 | 0.994980, 0.135912, 0.152961 |  | 0.1194 | 0.1691 | -0.0008 | 0.0000 | 0 |
| Anchored polynomial degree 4 | 0.994775, 0.141316, 0.128017, 0.026884 |  | 0.1194 | 0.1691 | -0.0007 | 0.0000 | 0 |

The Beta CDF is a useful production candidate because it is naturally bounded and monotone. The anchored polynomial is useful as a flexible descriptive reference, but it is less constrained structurally and should be treated as diagnostic unless monotonicity and stability are explicitly checked.

## Residual Distribution Around Reference Curves

Residuals are `actual cumulative pct - expected cumulative pct`. A good reference curve centers this distribution closer to zero and reduces asymmetric bias. The residual plot shows why the cumulative spend model should not be purely linear: the linear reference leaves a larger systematic position offset where the empirical spend curve is front-loaded.



## Stratification by Duration

Duration buckets have visibly different median cumulative spend curves. This supports the later duration-bucket Beta CDF model: the expected curve is not fully universal across short and long items.



## Stratification by Cluster Count

Cluster count is another proxy for payment cadence and lumpiness. Items with fewer clusters tend to show coarser jumps, while higher-cluster items provide smoother cumulative curves.



## Interpretation

The cumulative spend distribution is best understood as a conditional bounded distribution, not as a single ordinary marginal distribution. Its important properties are:

- Bounded support at `[0, 1]`, with a structural edge at 100% from final clusters.
- Strong positive relationship between elapsed percent and cumulative spend percent.
- A median curve that is not purely linear, indicating systematic front-loading in the clustered payment data.
- Wide conditional dispersion, especially in the middle of the lifecycle, caused by lumpy payment postings and differing payment cadence.
- Meaningful stratification by item duration and cluster count.

This motivates the next-stage notebook: fit expected-position curves and evaluate whether Beta CDF parameterizations outperform a transparent linear reference.